# Using OpenAi Python Client with Multiple LLM Providers

This notebook explores how to use an OpenAI-style Python client pattern across multiple LLM providers.

## Learning Goals

- Understand what an LLM provider is
- Learn how different providers can expose similar chat APIs
- Compare outputs from different models
- Build a clean provider-switching workflow
- Understand practical tradeoff such as speed, style, and cost

In [ ]:
%pip install openai python-dotenv

In [ ]:
import os
import time
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [ ]:
# Provider Config
OPENAI_MODEL = "gpt-4o-mini"

GROQ_BASE_URL = "https://api.groq.com/openai/v1"
GROQ_MODEL = "llama-3.3-70b-versatile"

In [ ]:
# Create clients
openai_client = OpenAI(api_key=OPENAI_API_KEY)
groq_client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url=GROQ_BASE_URL,
)


In [ ]:
# Unified Chat Function
def chat(client, model, prompt, system="You are a helpful assistant."):
    start = time.time()

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        temperature=0.3,
    )

    elapsed = round(time.time() - start, 3)

    return {
        "text": response.choices[0].message.content,
        "latency": elapsed,
        "tokens": response.usage.total_tokens if response.usage else None,
    }

In [ ]:
# prompt
prompt = "Explain embeddings in simple terms with an example"

In [ ]:
# openAI result
openai_result = chat(
    openai_client,
    OPENAI_MODEL,
    prompt
)

print("OpenAI Response:\n")
print(openai_result["text"])
print("\nLatency:", openai_result["latency"])
print("Tokens:", openai_result["tokens"])

In [ ]:
groq_result = chat(
    groq_client,
    GROQ_MODEL,
    prompt
)

print("Groq Response:\n")
print(groq_result["text"])
print("\nLatency:", groq_result["latency"])
print("Tokens:", groq_result["tokens"])

In [ ]:
# comparison
comparison = {
    "OpenAI": openai_result,
    "Groq": groq_result
}

comparison

## Model Comparison Insight

- Groq (llama-3.3-70b-versatile) provides very fast inference and good general responses.
- OpenAI models provide slightly more structured and consistent outputs.
- Using multiple providers allows optimization for latency vs quality.